# ARC Population Impact Estimator — L/M/H Intensity Zone Model

**American Red Cross — Storm Surge Mass Care Planning Tool**

This notebook estimates county-level population affected and impacted by hurricane storm surge,
classified into Low/Medium/High intensity zones per ARC Mass Care Planning Assumptions (V.6.0).

**How it works:**
1. **Input**: Building-level FAST damage predictions (CSV from Athena or upload)
2. **Step 1**: Classify each building into L/M/H intensity zone by surge depth
3. **Step 2**: Aggregate to county level: Population Affected & Impacted per zone
4. **Step 3**: Apply ARC conversion rates for shelter/feeding estimates

**Output**: CSV compatible with ARC Planning Assumptions Spreadsheet (columns J-R)

In [ ]:
# Install dependencies (run once)
!pip install pandas numpy openpyxl -q

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIGURATION — ARC Mass Care Planning Assumptions (V.6.0)
# ============================================================

# Intensity zone thresholds (Figure 9)
# Surge depth in feet at building location (Depth_Grid)
SURGE_THRESHOLDS = {'HIGH': 12, 'MEDIUM': 9, 'LOW': 4}  # feet

# Damage-based fallback thresholds (Figure 10)
# Used when surge depth is below LOW but structural damage is present
DAMAGE_THRESHOLDS = {'HIGH': 35, 'MEDIUM': 15, 'LOW': 0}  # BldgDmgPct (%)

# Mass care conversion rates (Figure 16)
# Applied to Population Impacted to estimate mass care needs
SHELTER_RATES = {'HIGH': 0.05, 'MEDIUM': 0.03, 'LOW': 0.01}
FEEDING_RATES = {'HIGH': 0.12, 'MEDIUM': 0.07, 'LOW': 0.03}

# Average household size (US Census Bureau)
AVG_HOUSEHOLD_SIZE = 2.53

print('Configuration loaded.')
print(f'  Surge thresholds: Low >= {SURGE_THRESHOLDS["LOW"]}ft, Med >= {SURGE_THRESHOLDS["MEDIUM"]}ft, High > {SURGE_THRESHOLDS["HIGH"]}ft')
print(f'  Shelter rates: Low={SHELTER_RATES["LOW"]:.0%}, Med={SHELTER_RATES["MEDIUM"]:.0%}, High={SHELTER_RATES["HIGH"]:.0%}')
print(f'  Feeding rates: Low={FEEDING_RATES["LOW"]:.0%}, Med={FEEDING_RATES["MEDIUM"]:.0%}, High={FEEDING_RATES["HIGH"]:.0%}')

## Step 1: Load Building-Level Damage Data

Upload your **building-level** FAST output CSV (from Athena query or pipeline).

Required columns:
- `FltyId`: Building identifier
- `Occ`: Occupancy type (e.g. RES1, COM1 — only RES* buildings are used)
- `Depth_Grid`: Surge depth at building location (feet)
- `Depth_in_Struc`: Water depth inside structure (feet)
- `BldgDmgPct`: Structural damage percentage (0-100)
- `county_fips5`: 5-digit county FIPS code

Optional columns:
- `BldgLossUSD`: Dollar loss estimate
- `Latitude` / `Longitude`: Building coordinates

In [ ]:
# ==========================================================
# OPTION A: Upload CSV in Google Colab
# ==========================================================
# from google.colab import files
# uploaded = files.upload()
# filename = list(uploaded.keys())[0]
# building_df = pd.read_csv(filename)

# ==========================================================
# OPTION B: Load from local file path
# ==========================================================
# building_df = pd.read_csv('fast_building_output.csv')

# ==========================================================
# OPTION C: Demo data — realistic Florida hurricane scenario
# ==========================================================
np.random.seed(42)
n_demo = 5000

# Simulate building-level FAST output for a Cat 3 hurricane hitting Tampa Bay area
county_pool = {
    '12057': ('Hillsborough', 1800),
    '12081': ('Manatee', 900),
    '12103': ('Pinellas', 1200),
    '12115': ('Sarasota', 800),
    '12021': ('Collier', 300),
}

rows = []
for fips, (name, n_bldgs) in county_pool.items():
    for i in range(n_bldgs):
        # Surge depths: exponential-ish distribution, most buildings get low surge
        surge = np.random.exponential(scale=5.0)
        surge = min(surge, 25.0)  # cap at 25 ft

        # Damage correlates with surge but with noise
        base_dmg = max(0, (surge - 2) * 6 + np.random.normal(0, 10))
        dmg_pct = np.clip(base_dmg, 0, 100)

        # Water inside structure depends on surge vs first-floor height
        first_floor_ht = np.random.choice([1, 2, 3, 4], p=[0.4, 0.3, 0.2, 0.1])
        depth_in_struc = max(0, surge - first_floor_ht)

        # Occupancy: 85% residential, 15% commercial
        occ = np.random.choice(
            ['RES1', 'RES2', 'RES3', 'COM1', 'COM2'],
            p=[0.55, 0.20, 0.10, 0.10, 0.05]
        )

        rows.append({
            'FltyId': f'{fips}_{i:05d}',
            'Occ': occ,
            'Depth_Grid': round(surge, 2),
            'Depth_in_Struc': round(depth_in_struc, 2),
            'BldgDmgPct': round(dmg_pct, 2),
            'BldgLossUSD': round(dmg_pct / 100 * np.random.uniform(150000, 500000), 0),
            'Latitude': round(np.random.uniform(26.5, 28.2), 5),
            'Longitude': round(np.random.uniform(-82.8, -81.5), 5),
            'county_fips5': fips,
        })

building_df = pd.DataFrame(rows)

print(f'Loaded {len(building_df):,} buildings across {building_df["county_fips5"].nunique()} counties')
print(f'  Residential: {building_df["Occ"].str.startswith("RES").sum():,}')
print(f'  Surge range: {building_df["Depth_Grid"].min():.1f} — {building_df["Depth_Grid"].max():.1f} ft')
print(f'  Damage range: {building_df["BldgDmgPct"].min():.1f} — {building_df["BldgDmgPct"].max():.1f}%')
building_df.head()

## Step 2: Classify Buildings into L/M/H Intensity Zones & Aggregate

Each residential building is classified into an intensity zone using two criteria:

1. **Primary — Surge depth** (`Depth_Grid`): High > 12 ft, Medium 9-12 ft, Low 4-8 ft (Figure 9)
2. **Fallback — Structural damage** (`BldgDmgPct`): High > 35%, Medium > 15%, Low > 0% (Figure 10)

Surge-based classification takes priority. The damage fallback captures buildings with significant
structural damage even when surge depth is below 4 ft (e.g., wind damage, wave action).

**Definitions** (per ARC):
- **Population Affected** = all residents in buildings within any intensity zone
- **Population Impacted** = subset with actual structural damage (`BldgDmgPct > 0` or `Depth_in_Struc > 0`)

In [ ]:
def classify_and_aggregate(df, surge_thresholds=SURGE_THRESHOLDS,
                           damage_thresholds=DAMAGE_THRESHOLDS,
                           avg_hh=AVG_HOUSEHOLD_SIZE):
    """Classify buildings into L/M/H intensity zones and aggregate to county level.

    Args:
        df: Building-level DataFrame with Occ, Depth_Grid, BldgDmgPct, etc.
        surge_thresholds: dict with HIGH/MEDIUM/LOW surge depth cutoffs (ft)
        damage_thresholds: dict with HIGH/MEDIUM/LOW damage pct cutoffs
        avg_hh: average household size for population estimation

    Returns:
        (county_wide_df, building_classified_df) tuple
    """
    # Filter residential buildings only
    res = df[df['Occ'].str.startswith('RES', na=False)].copy()
    print(f'  Residential buildings: {len(res):,} / {len(df):,} total')

    # Fill NaN surge/damage with 0
    res['Depth_Grid'] = res['Depth_Grid'].fillna(0)
    res['BldgDmgPct'] = res['BldgDmgPct'].fillna(0)
    res['Depth_in_Struc'] = res['Depth_in_Struc'].fillna(0)

    # Classify intensity zone: surge-primary, damage-fallback
    conditions = [
        res['Depth_Grid'] > surge_thresholds['HIGH'],
        res['Depth_Grid'] >= surge_thresholds['MEDIUM'],
        res['Depth_Grid'] >= surge_thresholds['LOW'],
        res['BldgDmgPct'] > damage_thresholds['HIGH'],
        res['BldgDmgPct'] > damage_thresholds['MEDIUM'],
        res['BldgDmgPct'] > damage_thresholds['LOW'],
    ]
    choices = ['HIGH', 'MEDIUM', 'LOW', 'HIGH', 'MEDIUM', 'LOW']
    res['intensity_zone'] = np.select(conditions, choices, default='NONE')

    # Report classification distribution
    zone_counts = res['intensity_zone'].value_counts()
    print(f'  Zone distribution:')
    for zone in ['HIGH', 'MEDIUM', 'LOW', 'NONE']:
        n = zone_counts.get(zone, 0)
        print(f'    {zone:6s}: {n:>6,} buildings ({n/len(res)*100:.1f}%)')

    # Filter out NONE (buildings not in any intensity zone)
    affected = res[res['intensity_zone'] != 'NONE'].copy()

    # Population estimate: 1 residential building = avg_hh persons
    affected['pop_estimate'] = avg_hh

    # Impacted flag: structural damage or water inside structure
    affected['is_impacted'] = (
        (affected['BldgDmgPct'] > 0) | (affected['Depth_in_Struc'] > 0)
    ).astype(int)

    # Aggregate by county x zone (long format)
    agg = affected.groupby(['county_fips5', 'intensity_zone']).agg(
        n_buildings_affected=('FltyId', 'count'),
        pop_affected=('pop_estimate', 'sum'),
        n_buildings_impacted=('is_impacted', 'sum'),
        avg_damage_pct=('BldgDmgPct', 'mean'),
        max_surge_ft=('Depth_Grid', 'max'),
    ).reset_index()

    # Compute pop_impacted from impacted building count
    agg['pop_impacted'] = agg['n_buildings_impacted'] * avg_hh

    # Sanity check: impacted <= affected
    agg['pop_impacted'] = agg[['pop_impacted', 'pop_affected']].min(axis=1)

    # Pivot to wide format: one row per county with L/M/H columns
    pivot_cols = ['pop_affected', 'pop_impacted', 'n_buildings_affected',
                  'n_buildings_impacted', 'avg_damage_pct', 'max_surge_ft']
    wide = agg.pivot_table(
        index='county_fips5',
        columns='intensity_zone',
        values=pivot_cols,
        fill_value=0,
    ).reset_index()

    # Flatten multi-level column names: ('pop_affected', 'HIGH') -> 'pop_affected_high'
    wide.columns = [
        '_'.join(col).strip('_').lower() if isinstance(col, tuple) and col[1] else col[0].lower()
        for col in wide.columns
    ]

    # Ensure all expected columns exist (some counties may lack a zone)
    for metric in ['pop_affected', 'pop_impacted', 'n_buildings_affected', 'n_buildings_impacted']:
        for zone in ['low', 'medium', 'high']:
            col = f'{metric}_{zone}'
            if col not in wide.columns:
                wide[col] = 0

    # Total columns
    wide['pop_affected_total'] = (
        wide['pop_affected_low'] + wide['pop_affected_medium'] + wide['pop_affected_high']
    )
    wide['pop_impacted_total'] = (
        wide['pop_impacted_low'] + wide['pop_impacted_medium'] + wide['pop_impacted_high']
    )

    print(f'\n  Aggregated to {len(wide)} counties')
    return wide, affected


# Run classification and aggregation
print('Classifying buildings into L/M/H intensity zones...')
county_results, building_classified = classify_and_aggregate(building_df)
county_results.head()

## Step 3: Apply ARC Mass Care Conversion Rates

ARC's Mass Care Planning Assumptions (V.6.0, Figure 16) provide standard conversion rates
from Population Impacted to mass care service needs:

| Impact Zone | Shelter % | Feeding % |
|-------------|-----------|-----------|
| **High**    | 5.0%      | 12.0%     |
| **Medium**  | 3.0%      | 7.0%      |
| **Low**     | 1.0%      | 3.0%      |

These rates are applied to **Population Impacted** (not Population Affected) per zone.

In [ ]:
# Apply ARC conversion rates to Population Impacted
for zone in ['low', 'medium', 'high']:
    zone_upper = zone.upper()
    imp_col = f'pop_impacted_{zone}'
    county_results[f'hh_shelter_{zone}'] = county_results[imp_col] * SHELTER_RATES[zone_upper]
    county_results[f'hh_feeding_{zone}'] = county_results[imp_col] * FEEDING_RATES[zone_upper]

# Totals
county_results['hh_shelter_total'] = (
    county_results['hh_shelter_low'] +
    county_results['hh_shelter_medium'] +
    county_results['hh_shelter_high']
)
county_results['hh_feeding_total'] = (
    county_results['hh_feeding_low'] +
    county_results['hh_feeding_medium'] +
    county_results['hh_feeding_high']
)

# ---- Summary Dashboard ----
n_counties = len(county_results)
pa_h = county_results['pop_affected_high'].sum()
pa_m = county_results['pop_affected_medium'].sum()
pa_l = county_results['pop_affected_low'].sum()
pa_t = pa_h + pa_m + pa_l

pi_h = county_results['pop_impacted_high'].sum()
pi_m = county_results['pop_impacted_medium'].sum()
pi_l = county_results['pop_impacted_low'].sum()
pi_t = pi_h + pi_m + pi_l

sh_h = county_results['hh_shelter_high'].sum()
sh_m = county_results['hh_shelter_medium'].sum()
sh_l = county_results['hh_shelter_low'].sum()
sh_t = sh_h + sh_m + sh_l

fd_h = county_results['hh_feeding_high'].sum()
fd_m = county_results['hh_feeding_medium'].sum()
fd_l = county_results['hh_feeding_low'].sum()
fd_t = fd_h + fd_m + fd_l

w = 56  # box width
print(f'\n{"":=<{w}}')
print(f'{"ARC MASS CARE PLANNING ESTIMATES":^{w}}')
print(f'{"":=<{w}}')
print(f'  Counties affected:         {n_counties:>10,}')
print(f'{"":_<{w}}')
print(f'  POPULATION AFFECTED')
print(f'    High zone (>{SURGE_THRESHOLDS["HIGH"]}ft):     {pa_h:>10,.0f}')
print(f'    Medium zone ({SURGE_THRESHOLDS["LOW"]}-{SURGE_THRESHOLDS["HIGH"]}ft):   {pa_m:>10,.0f}')
print(f'    Low zone ({SURGE_THRESHOLDS["LOW"]}-{SURGE_THRESHOLDS["MEDIUM"]}ft):     {pa_l:>10,.0f}')
print(f'    TOTAL:                   {pa_t:>10,.0f}')
print(f'{"":_<{w}}')
print(f'  POPULATION IMPACTED')
print(f'    High zone:               {pi_h:>10,.0f}')
print(f'    Medium zone:             {pi_m:>10,.0f}')
print(f'    Low zone:                {pi_l:>10,.0f}')
print(f'    TOTAL:                   {pi_t:>10,.0f}')
print(f'{"":_<{w}}')
print(f'  SHELTER ESTIMATES')
print(f'    High zone  ({SHELTER_RATES["HIGH"]:.0%}):       {sh_h:>10,.0f}')
print(f'    Medium zone ({SHELTER_RATES["MEDIUM"]:.0%}):     {sh_m:>10,.0f}')
print(f'    Low zone   ({SHELTER_RATES["LOW"]:.0%}):        {sh_l:>10,.0f}')
print(f'    TOTAL:                   {sh_t:>10,.0f}')
print(f'{"":_<{w}}')
print(f'  FEEDING ESTIMATES')
print(f'    High zone  ({FEEDING_RATES["HIGH"]:.0%}):      {fd_h:>10,.0f}')
print(f'    Medium zone ({FEEDING_RATES["MEDIUM"]:.0%}):    {fd_m:>10,.0f}')
print(f'    Low zone   ({FEEDING_RATES["LOW"]:.0%}):       {fd_l:>10,.0f}')
print(f'    TOTAL:                   {fd_t:>10,.0f}')
print(f'{"":=<{w}}')

In [ ]:
# Visualization: Stacked bar chart by county showing L/M/H population
try:
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    zone_colors = {'High': '#cc3333', 'Medium': '#ee8833', 'Low': '#3366cc'}

    counties = county_results['county_fips5'].values
    x = np.arange(len(counties))
    bar_width = 0.6

    # --- Left panel: Population Affected by zone ---
    ax = axes[0]
    bottom = np.zeros(len(counties))
    for zone, color in zone_colors.items():
        col = f'pop_affected_{zone.lower()}'
        vals = county_results[col].values
        ax.bar(x, vals, bar_width, bottom=bottom, label=zone, color=color, alpha=0.85)
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(counties, rotation=45, ha='right')
    ax.set_ylabel('Population')
    ax.set_title('Population Affected by Intensity Zone')
    ax.legend(title='Zone')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))

    # --- Right panel: Population Impacted by zone ---
    ax = axes[1]
    bottom = np.zeros(len(counties))
    for zone, color in zone_colors.items():
        col = f'pop_impacted_{zone.lower()}'
        vals = county_results[col].values
        ax.bar(x, vals, bar_width, bottom=bottom, label=zone, color=color, alpha=0.85)
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(counties, rotation=45, ha='right')
    ax.set_ylabel('Population')
    ax.set_title('Population Impacted by Intensity Zone')
    ax.legend(title='Zone')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))

    plt.suptitle('ARC Mass Care Planning: County-Level L/M/H Intensity Zone Estimates',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

except ImportError:
    print('matplotlib not available — skipping visualization.')

## Step 4: Export Results

Export two files:
1. **CSV** — columns matching ARC Planning Assumptions Spreadsheet (J-R)
2. **Excel** — two sheets: "Estimates" (county data) and "Parameters" (model configuration)

In [ ]:
# Select and order columns matching Planning Assumptions Spreadsheet
export_cols = [
    'county_fips5',
    # Columns J/K/L: Population Affected
    'pop_affected_low', 'pop_affected_medium', 'pop_affected_high',
    # Columns M/N/O: Population Impacted
    'pop_impacted_low', 'pop_impacted_medium', 'pop_impacted_high',
    # Columns P/Q/R: Households Needing Shelter
    'hh_shelter_low', 'hh_shelter_medium', 'hh_shelter_high',
    # Supplementary: feeding estimates
    'hh_feeding_low', 'hh_feeding_medium', 'hh_feeding_high',
    # Totals
    'pop_affected_total', 'pop_impacted_total',
    'hh_shelter_total', 'hh_feeding_total',
]
available_cols = [c for c in export_cols if c in county_results.columns]
export_df = county_results[available_cols].copy()

# Fill any NaN with 0
export_df = export_df.fillna(0)

# Round to integers for population counts
int_cols = [c for c in export_df.columns if c != 'county_fips5']
export_df[int_cols] = export_df[int_cols].round(0).astype(int)

# --- CSV ---
csv_path = 'planning_assumptions_output.csv'
export_df.to_csv(csv_path, index=False)
print(f'CSV exported: {csv_path} ({len(export_df)} rows, {len(export_df.columns)} columns)')

# --- Excel with two sheets ---
xlsx_path = 'arc_planning_estimates.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    export_df.to_excel(writer, sheet_name='Estimates', index=False)

    # Parameters sheet: document all configuration used
    params = pd.DataFrame({
        'Parameter': [
            'Surge Threshold — High (ft)',
            'Surge Threshold — Medium (ft)',
            'Surge Threshold — Low (ft)',
            'Damage Threshold — High (%)',
            'Damage Threshold — Medium (%)',
            'Damage Threshold — Low (%)',
            'Shelter Rate — High',
            'Shelter Rate — Medium',
            'Shelter Rate — Low',
            'Feeding Rate — High',
            'Feeding Rate — Medium',
            'Feeding Rate — Low',
            'Avg Household Size',
            'Classification Method',
            'Reference Document',
            'Model Version',
        ],
        'Value': [
            f'> {SURGE_THRESHOLDS["HIGH"]}',
            f'>= {SURGE_THRESHOLDS["MEDIUM"]}',
            f'>= {SURGE_THRESHOLDS["LOW"]}',
            f'> {DAMAGE_THRESHOLDS["HIGH"]}',
            f'> {DAMAGE_THRESHOLDS["MEDIUM"]}',
            f'> {DAMAGE_THRESHOLDS["LOW"]}',
            f'{SHELTER_RATES["HIGH"]:.0%}',
            f'{SHELTER_RATES["MEDIUM"]:.0%}',
            f'{SHELTER_RATES["LOW"]:.0%}',
            f'{FEEDING_RATES["HIGH"]:.0%}',
            f'{FEEDING_RATES["MEDIUM"]:.0%}',
            f'{FEEDING_RATES["LOW"]:.0%}',
            str(AVG_HOUSEHOLD_SIZE),
            'Surge-primary, damage-fallback (np.select)',
            'ARC Mass Care Planning Assumptions V.6.0',
            'v2.0-LMH (2026-03-10)',
        ],
    })
    params.to_excel(writer, sheet_name='Parameters', index=False)

print(f'Excel exported: {xlsx_path} (sheets: Estimates, Parameters)')

# Download in Colab
try:
    from google.colab import files
    files.download(csv_path)
    files.download(xlsx_path)
except ImportError:
    pass

export_df

---

## Model Card

**Model**: ARC Population Impact Estimator v2.0 (L/M/H Intensity Zone)

**Approach**: Deterministic classification of FEMA FAST building-level damage predictions into
Low/Medium/High intensity zones, per ARC Mass Care Planning Assumptions V.6.0. No ML models
are used — this is a rules-based classification and aggregation pipeline.

**Classification Logic** (per building):
1. **Surge-primary**: `Depth_Grid > 12ft` = High, `>= 9ft` = Medium, `>= 4ft` = Low
2. **Damage-fallback**: `BldgDmgPct > 35%` = High, `> 15%` = Medium, `> 0%` = Low
3. Surge classification takes priority; damage-fallback captures wind/wave damage below 4ft surge

**Population Estimation**:
- Each residential building (Occ = RES*) represents ~2.53 persons (US Census avg household size)
- **Population Affected**: all residential buildings classified into any L/M/H zone
- **Population Impacted**: subset with `BldgDmgPct > 0` or `Depth_in_Struc > 0`

**Mass Care Conversion Rates** (ARC Figure 16):
- Shelter: High = 5%, Medium = 3%, Low = 1% of Population Impacted
- Feeding: High = 12%, Medium = 7%, Low = 3% of Population Impacted

**Output Format**: CSV columns match ARC Planning Assumptions Spreadsheet (J-R):
Population Affected (L/M/H), Population Impacted (L/M/H), Households Needing Shelter (L/M/H)

**Data Source**: FEMA FAST (Flood Assessment Structure Tool) building-level predictions,
using NSI (National Structure Inventory) building stock with SLOSH storm surge rasters.

**Key References**:
- ARC Mass Care Planning Assumptions Job Tool V.6.0 (Figures 9, 10, 16, 17)
- FEMA FAST Technical Documentation
- US Census Bureau: Average Household Size (2.53)

**Known Limitations**:
1. Population estimated from building count x 2.53, not actual census population at building level.
   NSI nighttime population fields (pop2pmu65/pop2pmo65) are used when available in the Athena pipeline.
2. Surge thresholds (4/9/12 ft) are based on ARC guidance and have not yet been empirically
   optimized against historical events. Threshold sensitivity analysis is recommended.
3. Only covers storm surge impact zones modeled by FAST — does not capture inland rainfall
   flooding, which can drive significant displacement (e.g., Florence 2018).
4. Damage-based fallback classification may double-count buildings near surge threshold
   boundaries in edge cases.
5. Commercial and institutional buildings (COM*, IND*) are excluded; their occupants
   may still require mass care services.

**Version History**:
- v1.0 (2026-03-01): Single shelter rate model (0.73% of displaced)
- v2.0 (2026-03-10): L/M/H intensity zone model aligned with ARC Planning Assumptions V.6.0